# PDP-CR · Chạy TSR tích hợp phép dời tác vụ từ đầu

Chọn runtime **CPU** rồi chạy ô benchmark. Notebook chạy riêng **TSR tích hợp**, thử dời một tác vụ găng từ vòng lặp đầu tiên.

Ba instance **LC101-n50, LR101-n50 và LRC101-n50** giữ depot và tác vụ ID 1–50 từ dữ liệu của Thảo. Mỗi instance dùng nghiệm greedy và chạy song song năm seed **0–4**, với **600 giây CPU mỗi seed**. Tổng cộng có **15 lượt**.

ZIP kết quả chứa `runs.csv` theo seed, `summary.csv` theo instance, `anytime.csv` và các tệp JSON chi tiết. Các kết quả này có thể ghép với kết quả cũ theo instance và seed. Bật **SAVE_TO_DRIVE** trước khi chạy để lưu các seed đã hoàn thành và tiếp tục với cùng cấu hình.


In [ ]:
# @title Chạy benchmark PDP-CR
TIME_LIMIT = 600  # @param {type:"number"}
SEEDS = [0, 1, 2, 3, 4]  # @param {type:"raw"}
SAVE_TO_DRIVE = False  # @param {type:"boolean"}

from io import BytesIO
from pathlib import Path
from runpy import run_path
from urllib.request import urlopen
from zipfile import ZipFile

from google.colab import drive, files

REVISION = "879e54500d989c8936dba3b59fb7493a1d6ef3e5"
workspace = Path("/content/pdpcr_colab")
source = workspace / f"colab_test-{REVISION}"
runner = source / "run_tsr_colab.py"
if not runner.is_file():
    print("Đang tải mã benchmark từ GitHub...", flush=True)
    url = f"https://codeload.github.com/dhducpro/colab_test/zip/{REVISION}"
    with urlopen(url, timeout=60) as response, ZipFile(BytesIO(response.read())) as archive:
        archive.extractall(workspace)

output_root = workspace / "results"
if SAVE_TO_DRIVE:
    drive.mount("/content/drive")
    output_root = Path("/content/drive/MyDrive/PDPCR_TSR_benchmark")

benchmark = run_path(str(runner))
result_zip = benchmark["run_colab"](
    TIME_LIMIT, SEEDS, benchmark["INTEGRATED"], output_root, REVISION,
    instances=benchmark["N50_INSTANCES"], integrated_only=True,
)
files.download(str(result_zip))
